In [4]:
import pandas as pd

df = pd.read_excel(r'C:\Users\bette\OneDrive\Desktop\streaming_churn_raw.xlsx')

print(df.shape)
df.head()

(5000, 14)


,customer_id,age,gender,subscription_type,watch_hours,avg_watch_time_per_day,last_login_days,region,device,monthly_fee,churned,payment,number_of_profiles,favorite_genre
0,fdf130c6-cd3c-434b-abbf-4645c8ca928d,18,Male,Standard,03:33:36,03:36:00,23,Oceania,Tablet,13.99,0,Debit Card,4,Horror
1,7fdace55-3767-41d2-85d3-e43a5eddede6,18,Female,Basic,1900-01-01 11:31:12,04:36:00,40,South America,Tablet,8.99,0,Crypto,3,Action
2,8af34fa6-55d3-4e2f-bb67-94d27ba89512,18,Other,Standard,16:48:36,05:36:00,56,Oceania,Tablet,13.99,1,Credit Card,2,Drama
3,ce63b3cf-0d7a-499f-a1d7-5ee3a26a0257,18,Female,Premium,06:48:00,06:36:00,20,South America,Laptop,17.99,0,Credit Card,4,Drama
4,533ad58f-7675-4e1f-8a7b-27f64e6ac546,18,Other,Premium,17:45:00,07:36:00,9,Oceania,Desktop,17.99,0,Gift Card,1,Comedy


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             5000 non-null   object 
 1   age                     5000 non-null   int64  
 2   gender                  5000 non-null   object 
 3   subscription_type       5000 non-null   object 
 4   watch_hours             4999 non-null   object 
 5   avg_watch_time_per_day  5000 non-null   object 
 6   last_login_days         5000 non-null   int64  
 7   region                  5000 non-null   object 
 8   device                  5000 non-null   object 
 9   monthly_fee             5000 non-null   float64
 10  churned                 5000 non-null   int64  
 11  payment                 5000 non-null   object 
 12  number_of_profiles      5000 non-null   int64  
 13  favorite_genre          5000 non-null   object 
dtypes: float64(1), int64(4), object(9)
memor

In [21]:
# find the row with the missing watch_hours
df[df['watch_hours'].isnull()]

,customer_id,age,gender,subscription_type,watch_hours,avg_watch_time_per_day,last_login_days,region,device,monthly_fee,churned,payment,number_of_profiles,favorite_genre
2916,8d755725-4f16-4006-bdae-fa3dc9fa2579,49,Female,Standard,NaN,07:36:00,27,Asia,TV,13.99,0,Debit Card,3,Action


In [23]:
# check for duplicate customers
df['customer_id'].duplicated().sum()

0

In [25]:
def time_to_hours(val):
    s = str(val)
    if ' ' in s:
        s = s.split(' ')[1]
    h, m, sec = s.split(':')
    return int(h) + int(m)/60 + int(sec)/3600

df['avg_watch_temp'] = df['avg_watch_time_per_day'].apply(time_to_hours)

df['avg_watch_temp'].diff().head(30)

0      NaN
1      1.0
2      1.0
3      1.0
4      1.0
5      1.0
6      1.0
7      1.0
8      1.0
9      1.0
10     1.0
11     1.0
12     1.0
13     1.0
14     1.0
15     1.0
16     1.0
17     1.0
18     1.0
19     1.0
20     1.0
21   -23.0
22     1.0
23     1.0
24     1.0
25     1.0
26     1.0
27     1.0
28     1.0
29     1.0
Name: avg_watch_temp, dtype: float64

In [27]:
# drop the temporary test column and the broken real column
df = df.drop(columns=['avg_watch_temp', 'avg_watch_time_per_day'])

# drop the 1 row with missing watch_hours
df = df.dropna(subset=['watch_hours'])

print(df.shape)

(4999, 13)


In [29]:
def parse_watch_hours(val):
    s = str(val)
    if ' ' in s:          # strip off any '1900-01-01 ' date prefix
        s = s.split(' ')[1]
    h, m, sec = s.split(':')
    return round(int(h) + int(m)/60 + int(sec)/3600, 2)

df['watch_hours'] = df['watch_hours'].apply(parse_watch_hours)

df['watch_hours'].head(10)

0     3.56
1    11.52
2    16.81
3     6.80
4    17.75
5    15.17
6     0.97
7    12.42
8    17.28
9    14.18
Name: watch_hours, dtype: float64

In [33]:
df['watch_hours'].dtype

dtype('float64')

In [35]:
# check the range makes sense (should be between 0 and 24, since it's hours in a day)
print(df['watch_hours'].min(), df['watch_hours'].max())

# check monthly_fee is consistent with subscription_type (no weird mismatched pricing)
df.groupby('subscription_type')['monthly_fee'].unique()

0.01 23.97


subscription_type
Basic        [8.99]
Premium     [17.99]
Standard    [13.99]
Name: monthly_fee, dtype: object

In [37]:
# double check no more nulls anywhere
df.isnull().sum()

customer_id           0
age                   0
gender                0
subscription_type     0
watch_hours           0
last_login_days       0
region                0
device                0
monthly_fee           0
churned               0
payment               0
number_of_profiles    0
favorite_genre        0
dtype: int64

In [45]:
df.to_csv('netflix_churn_clean.csv', index=False)